In [1]:
from google.colab import files
uploaded = files.upload()

import zipfile

zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

Saving archive (7).zip to archive (7).zip


In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

In [3]:
import os

base_path = "/content/dataset"

for root, dirs, files in os.walk(base_path):
    print("📂", root)
    for d in dirs:
        print("   └──", d)

📂 /content/dataset
   └── rice_leaf_diseases
📂 /content/dataset/rice_leaf_diseases
   └── Brown spot
   └── Bacterial leaf blight
   └── Leaf smut
📂 /content/dataset/rice_leaf_diseases/Brown spot
📂 /content/dataset/rice_leaf_diseases/Bacterial leaf blight
📂 /content/dataset/rice_leaf_diseases/Leaf smut


In [ ]:
IMG_SIZE = 224

data = []
labels = []

dataset_path = "/content/dataset/rice_leaf_diseases"

for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)

    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            try:
                img = cv2.imread(img_path)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img / 255.0

                data.append(img)
                labels.append(class_name)
            except:
                pass

data = np.array(data)
labels = np.array(labels)

print("Classes:", set(labels))
print("Total images:", len(data))

Classes: {np.str_('Bacterial leaf blight'), np.str_('Brown spot'), np.str_('Leaf smut')}
Total images: 120


In [ ]:
import numpy as np

# Get unique classes
unique_classes = list(set(labels))

# Create mapping (label → number)
label_map = {}
for i, label in enumerate(unique_classes):
    label_map[label] = i

print("Label Map:", label_map)

# Convert labels to numbers
labels_encoded = []
for label in labels:
    labels_encoded.append(label_map[label])

labels_encoded = np.array(labels_encoded)

# One-hot encoding manually
num_classes = len(unique_classes)

labels_categorical = np.zeros((len(labels_encoded), num_classes))

for i in range(len(labels_encoded)):
    labels_categorical[i][labels_encoded[i]] = 1

Label Map: {np.str_('Bacterial leaf blight'): 0, np.str_('Brown spot'): 1, np.str_('Leaf smut'): 2}


In [ ]:
import numpy as np

# Shuffle data
indices = np.arange(len(data))
np.random.shuffle(indices)

data = data[indices]
labels_categorical = labels_categorical[indices]

# Split ratio
split_ratio = 0.8
split_index = int(len(data) * split_ratio)

# Split
X_train = data[:split_index]
X_test = data[split_index:]

y_train = labels_categorical[:split_index]
y_test = labels_categorical[split_index:]

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 96
Test size: 24


In [ ]:
import random

def augment_image(img):
    # Flip
    if random.random() > 0.5:
        img = cv2.flip(img, 1)

    # Rotation
    angle = random.randint(-20, 20)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
    img = cv2.warpAffine(img, M, (w, h))

    return img

In [ ]:
aug_images = []
aug_labels = []

for i in range(len(X_train)):
    img = augment_image(X_train[i])
    aug_images.append(img)
    aug_labels.append(y_train[i])

X_train = np.concatenate((X_train, np.array(aug_images)))
y_train = np.concatenate((y_train, np.array(aug_labels)))

print("After augmentation:", len(X_train))

After augmentation: 192


In [ ]:
model = tf.keras.models.Sequential()

model.add(tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)))
model.add(tf.keras.layers.MaxPooling2D(2,2))

model.add(tf.keras.layers.Conv2D(64, (3,3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(2,2))

model.add(tf.keras.layers.Conv2D(128, (3,3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(2,2))

model.add(tf.keras.layers.Flatten())

model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dropout(0.5))

model.add(tf.keras.layers.Dense(num_classes, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32
)

Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 23s 4s/step - accuracy: 0.4427 - loss: 1.1181 - val_accuracy: 0.5833 - val_loss: 0.9063
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 0.4948 - loss: 1.0016 - val_accuracy: 0.5000 - val_loss: 0.8706
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 24s 4s/step - accuracy: 0.5833 - loss: 0.8730 - val_accuracy: 0.6667 - val_loss: 0.7182
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 39s 4s/step - accuracy: 0.7083 - loss: 0.8074 - val_accuracy: 0.7083 - val_loss: 0.6411
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 42s 4s/step - accuracy: 0.6771 - loss: 0.7167 - val_accuracy: 0.6667 - val_loss: 0.6386
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 23s 4s/step - accuracy: 0.6875 - loss: 0.6807 - val_accuracy: 0.8750 - val_loss: 0.4592
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 21s 4s/step - accuracy: 0.7188 - loss: 0.5863 - val_accuracy: 0.9167 - val_loss: 0.3378
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 23s 4s/step - accuracy: 0.7812 - loss: 0.4688 - val_accuracy: 0.7083 - val_loss: 0.5800
Epoch 9/

In [ ]:
y_pred = model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


In [ ]:
correct = 0

for i in range(len(y_true)):
    if y_true[i] == y_pred_classes[i]:
        correct += 1

accuracy = correct / len(y_true)

print("Accuracy:", accuracy)

Accuracy: 0.7916666666666666


In [ ]:
num_classes = len(unique_classes)

precision_list = []
recall_list = []

for c in range(num_classes):
    TP = FP = FN = 0

    for i in range(len(y_true)):
        if y_pred_classes[i] == c and y_true[i] == c:
            TP += 1
        elif y_pred_classes[i] == c and y_true[i] != c:
            FP += 1
        elif y_pred_classes[i] != c and y_true[i] == c:
            FN += 1

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    precision_list.append(precision)
    recall_list.append(recall)

print("Precision:", np.mean(precision_list))
print("Recall:", np.mean(recall_list))

Precision: 0.8296296284337448
Recall: 0.8333333321944444
